# Lab 9: A smaller embedding model for search

Search over the banking messages, built three times: with a large teacher, with a small student
as published, and with that student after distillation.

1. Embed the collection and the queries with the teacher.
2. Do the same with the published student. This is the baseline.
3. Distil the student to match the teacher's similarities. No labels are used.
4. Score all three, and compare speed and storage.
5. Store the student's embeddings, search them, and measure what an index misses.

This lab stands alone. Allow about 30 minutes. Postgres with pgvector is optional: set
`KD_COURSE_DATABASE_URL` before starting Jupyter to use it.

In [ ]:
from kd_course.env import check_environment

report = check_environment()
print(report.summary())
assert report.ok, "Fix the problems listed above before continuing."

BATCH_SIZE = 128  # texts per training step; each step compares every pair in the batch
K = 10  # results per query

## 1. The data

The collection is the pool. The queries come from the test set, so no query is in the collection.
The messages' types are used only to score results, never to train.

In [ ]:
from kd_course.data import load_banking77, make_splits

bundle = load_banking77()
splits = make_splits(bundle.train, bundle.test)
corpus, corpus_labels = splits.pool.texts, splits.pool.labels
queries, query_labels = splits.test.texts, splits.test.labels
print(f"{len(corpus)} messages in the collection, {len(queries)} queries")

## 2. One function to embed, time and score a model

In [ ]:
import gc
import time

import torch

from kd_course.embeddings import (
    embed_texts,
    neighbour_overlap,
    per_query_precision,
    retrieval_scores,
    top_neighbours,
)

results = {}


def evaluate(name, model, tokenizer):
    start = time.perf_counter()
    corpus_vectors = embed_texts(model, tokenizer, corpus)
    rate = len(corpus) / (time.perf_counter() - start)
    query_vectors = embed_texts(model, tokenizer, queries)
    found = top_neighbours(query_vectors, corpus_vectors, k=K)
    scores = retrieval_scores(found, query_labels, corpus_labels)
    megabytes = corpus_vectors.shape[0] * corpus_vectors.shape[1] * 4 / 1_048_576
    results[name] = {
        "found": found,
        "precision": per_query_precision(found, query_labels, corpus_labels),
        "vectors": corpus_vectors,
        "query_vectors": query_vectors,
        "row": (scores["precision_at_k"], scores["top_1_same_class"], rate, megabytes),
    }
    print(
        f"{name}: precision at {K} {scores['precision_at_k']:.1%}, "
        f"top result same type {scores['top_1_same_class']:.1%}"
    )


def free():
    """Release GPU memory. Delete your own references to a model before calling this."""
    gc.collect()
    torch.cuda.empty_cache()

## 3. The teacher

In [ ]:
from kd_course.embeddings import EMBED_STUDENT_ID, EMBED_TEACHER_ID, load_embedder

teacher, teacher_tokenizer = load_embedder(EMBED_TEACHER_ID)
evaluate("teacher", teacher, teacher_tokenizer)
teacher_vectors = results["teacher"]["vectors"]
del teacher
free()

## 4. The student, as published

No training. This is the baseline.

In [ ]:
student, student_tokenizer = load_embedder(EMBED_STUDENT_ID)
evaluate("student, as published", student, student_tokenizer)

## 5. Distil the student

Each step takes a batch of messages and compares the student's similarity for every pair with the
teacher's. Nothing here uses a label.

In [ ]:
from kd_course.embeddings import train_embedding_student

history = train_embedding_student(
    student, student_tokenizer, corpus, teacher_vectors, epochs=3, batch_size=BATCH_SIZE
)
print(history)
evaluate("student, distilled", student, student_tokenizer)
del student
free()

## 6. Compare

In [ ]:
from kd_course.metrics import paired_mean_difference

print(f"{'':<24}{'precision':>10}{'top result':>12}{'texts/s':>10}{'collection':>12}")
for name, result in results.items():
    precision, top_1, rate, megabytes = result["row"]
    print(f"{name:<24}{precision:>10.1%}{top_1:>12.1%}{rate:>10,.0f}{megabytes:>9.1f} MB")

print()
for name in ("student, as published", "student, distilled"):
    overlap = neighbour_overlap(results[name]["found"], results["teacher"]["found"])
    print(f"{name:<24} shares {overlap:.0%} of its top {K} with the teacher")


def report(title, a, b):
    r = paired_mean_difference(results[a]["precision"], results[b]["precision"])
    print(
        f"{title}: {r.difference * 100:+.1f} points "
        f"(95% interval {r.low * 100:+.1f} to {r.high * 100:+.1f})"
    )


print()
report("Distilled minus published ", "student, as published", "student, distilled")
report("Distilled minus teacher   ", "teacher", "student, distilled")

## 7. Store and search

The distilled student's embeddings go into a store. With `KD_COURSE_DATABASE_URL` set this is a
Postgres table using pgvector; otherwise it is an in-memory store with the same methods.

In [ ]:
from kd_course.vector_store import MemoryStore, open_store

vectors = results["student, distilled"]["vectors"]
store = open_store(dim=vectors.shape[1], table="kd_lab9_messages")
in_memory = isinstance(store, MemoryStore)
print("Using an in-memory store." if in_memory else "Using Postgres with pgvector.")

store.create()
store.add(list(range(len(corpus))), corpus, vectors.tolist(), corpus_labels)
print(f"{store.count()} rows, {store.size_bytes() / 1_048_576:.1f} MB before indexing")

## 8. Exact search against the index

Two hundred queries are run with no index, which compares each query with every row. Then the
index is built and they are run again, at the default search effort and at a higher one. The share
of exact search's results that the index returns is its recall.

In [ ]:
from kd_course.vector_store import index_recall

query_vectors = results["student, distilled"]["query_vectors"][:200].tolist()


def timed_search(search_effort=None):
    start = time.perf_counter()
    found = [store.search(q, k=K, search_effort=search_effort) for q in query_vectors]
    return found, (time.perf_counter() - start) / len(query_vectors) * 1000


exact, exact_ms = timed_search()
print(f"Exact search:            {exact_ms:.1f} ms per query")

store.create_index()
for effort in (None, 200):
    found, ms = timed_search(effort)
    name = "default" if effort is None else str(effort)
    print(
        f"Index, effort {name:<8}  {ms:.1f} ms per query, recall {index_recall(exact, found):.1%}"
    )

print(f"{store.size_bytes() / 1_048_576:.1f} MB after indexing")
if in_memory:
    print("The in-memory store has no index, so every run above is the same exact search.")

## 9. What did you see?

Double-click this cell and write your answers.

**a. How far behind the teacher was the published student? Was that more or less than you predicted?**

**b. Did overlap with the teacher rise after distillation? Did precision? If they moved differently, explain why using lesson 9.3.**

**c. How many times faster did the student embed text, and how much smaller was its collection? For 50 million documents, what would those two figures mean?**

**d. What share of exact search's results did the index return at the default effort, and at the higher one? What did the higher effort cost in time? On 9,000 rows the index is not needed; at what size would you expect it to matter?**

**e. Would you distil for this domain, or use the published student as it is? What result would change your mind?**